# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [ ]:

import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

## Raven test run

In [ ]:
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "CypressRiverT_ERA5"

In [ ]:
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

## Calibration set-up



In [ ]:
calibrator = spot_setup_rvn(raven_name, DATA_CAL, obj_fn= "nse", split_areas = False, spring_eval= False)

# calibrator.dep_fact = Uniform(low=0.80, high=1.20)
# calibrator.seep_k = Uniform(low=0.00, high=0.10)
# calibrator.basin_fact = Uniform(low=0.80, high=1.20)

# sim_results = calibrator.simulation([1,0.05,1])
# obs_results = calibrator.evaluation()

In [ ]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.rope(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='rope_rv_nse_nosplit',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



In [ ]:
rep = 30
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

In [ ]:
results_df = sp.analyser.load_csv_results(sampler.dbname)

In [ ]:
results_df = pd.DataFrame(results_df)

In [ ]:
param_nse_sets = results_df[results_df.columns[0:4]]
np.corrcoef(param_nse_sets.transpose())

In [ ]:
from sklearn.metrics import mutual_info_score

param_nse_sets_norm = param_nse_sets.copy()
param_nse_sets_norm.iloc[:, 0] = ((param_nse_sets_norm.iloc[:, 0] - param_nse_sets_norm.iloc[:, 0].min()) \
                                  / (param_nse_sets_norm.iloc[:, 0].max() - param_nse_sets_norm.iloc[:, 0].min())).round(1)
param_nse_sets_norm.iloc[:, 1] = ((param_nse_sets_norm.iloc[:, 1] - 0.8) / 0.4).round(1)
param_nse_sets_norm.iloc[:, 2] = (param_nse_sets_norm.iloc[:, 2]/0.1).round(1)
param_nse_sets_norm.iloc[:, 3] = ((param_nse_sets_norm.iloc[:, 3] - 0.8) / 0.4).round(1)

mi_dep = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 1])
mi_seep = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 2])
mi_area = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 3])

print(f"Mutual Information between dep_fact and nse: {mi_dep}")
print(f"Mutual Information between seep_k and nse: {mi_seep}")
print(f"Mutual Information between basin_area and nse: {mi_area}")

In [ ]:
param_nse_sets_norm

## Additional workflows

- Extract Raven results

- Specify calibration parameters, method, iterations, etc.
- Run calibration
- Store results & optimal parameters

## Update Raven files

In [ ]:
Cypress_rvp = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvp")
Cypress_rvp.rvp_update("base", np.array([0.89955,0.09,0.818696]))

In [ ]:
Cypress_rvh = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvh")

Cypress_rvh.rvh_update("base", np.array([0.89955,0.09,0.818696]))

In [ ]:
rv.run_raven(raven_name, DATA_CAL)

Manual Raven results with same parameters yield a different NSE

In [ ]:
simulation = pd.read_csv(os.path.join(DATA_CAL, "output/"+raven_name+"_Hydrographs.csv"))["sub875 [m3/s]"]

evaluation = pd.read_csv(os.path.join(DATA_CAL, "output/"+raven_name+"_Hydrographs.csv"))["sub875 (observed) [m3/s]"]

nse = sp.objectivefunctions.nashsutcliffe(simulation, evaluation)

nse = np.squeeze(nse).item()

print(f"Nash-Sutcliffe Efficiency (NSE): {nse}")

custom_nse = 1- (np.sum((simulation - evaluation)**2) / np.sum((evaluation - np.mean(evaluation))**2))
print(f"Custom NSE: {custom_nse}")


In [ ]:
kge = sp.objectivefunctions.kge(simulation, evaluation)

print(kge)